# 异地标准化训练（笔记本版）

这台服务器只有 Jupyter，所以体检、训练、回传都放在这个笔记本里。你**从上到下依次运行**即可
（菜单 `Kernel -> Restart & Run All`，或逐格 `Shift+Enter`）。

三条规矩：

1. **不要修改任何脚本文件**，不要动训练包里的任何文件。
2. 你只有第 1 节需要改（两个路径 + 两个开关）。
3. 屏幕上任何 `!!!!` 方框都是说给你看的，里面有四段：发生了什么 / 为什么是问题 / 下一步做什么 / 找谁。

如果你把「体检」那一格跑出红字，**不要往下跑**，把内容发回本机。

> 注意：Jupyter 的 `Run All` **不会**因为某一格报错就停下来，它会继续跑后面的格子。
> 所以后面每一格都会**自己再检查一次闸门**：体检没通过时，它们会拒绝执行并各自报错。
> 看到「体检没有通过，本格拒绝执行」就说明闸门在起作用，这是正常的。


In [ ]:
# ===== 1. 配置：你只需要改这一格 =====

# 解压后的训练包目录（就是含 MANIFEST.json 的那个目录）。留空则下面会自动找。
BUNDLE = ""

# 工作目录：试次 / 日志 / 指纹 / 报告都落在这里。会越长越大，注意磁盘空间。
WORK_DIR = "_pt_work"

# 真训练开关。默认 False —— 只做体检和冒烟，不产模型。确认要真训再改 True。
DO_REAL_TRAIN = False

# 超参搜索开关。默认 False —— 第 7 节只打印计划、不真跑。
# 只有本机随包发了网格时第 7 节才有事做；没有网格时这一格会被跳过，改不改它都一样。
DO_SEARCH = False

# 回传里要不要带模型权重。默认 False —— 搜索场景只看指标，权重又大又用不上。
# 周训要上线时才需要带，**本机会在交付说明里明确让你打开它**，没说你就不动。
WITH_MODELS = False

# 体检里出现「判不了」时，必须先把那几条读完，再把这里改成 True 才允许继续。
# 判不了 不等于 通过。
I_READ_THE_UNKNOWNS = False

# 闸门状态。由第 4 节体检决定，你不要手改这两行。
GATE_OK = False
DOCTOR_VERDICT = None


In [ ]:
# ===== 2. 装载（不用改，直接跑）=====
import os, sys, json, glob, time, zipfile, shutil
from pathlib import Path

os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")   # 禁落 __pycache__


def _find_pt_root():
    here = Path.cwd()
    for cand in (here / "attest", here, here.parent / "attest"):
        if (cand / "pt.py").is_file() and (cand / "scripts" / "pt_doctor.py").is_file():
            return cand.resolve()
    for hit in sorted(Path.cwd().rglob("pt.py")):
        if (hit.parent / "scripts" / "pt_doctor.py").is_file():
            return hit.parent.resolve()
    return None


PT_ROOT = _find_pt_root()
assert PT_ROOT, ("找不到 attest 目录（里面应当有 pt.py 和 scripts/）。"
                 "请把 attest 文件夹和训练包一起上传到当前目录。")

sys.path.insert(0, str(PT_ROOT / "scripts"))
import pt_common as C, pt_doctor, pt_train, pt_eval, pt_search, pt_pack   # noqa: E402

C.setup_console()
print("attest :", PT_ROOT)
print("python         :", sys.executable)
print("工作目录       :", Path.cwd())


In [ ]:
# ===== 3. 找到包（会自动解压当且仅当只找到一个 .zip）=====
# 笔记本本身常常就在 attest/ 里，所以往上看一层、两层再找包。
sys.path.insert(0, str(PT_ROOT / "scripts"))
import pt_common as _c
_SEARCH_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]


def _walk_zips_and_bundles():
    zips, bundles = [], []
    for r in _SEARCH_ROOTS:
        if not r.is_dir():
            continue
        for d in sorted(r.iterdir()):
            try:
                if d.is_file() and d.suffix.lower() == ".zip":
                    zips.append(d)
                elif (d.is_dir() and d.name != "attest"
                      and (d / "MANIFEST.json").is_file()):
                    bundles.append(d)
            except OSError:
                pass
    return zips, bundles


def _dedup(seq):
    seen, out = set(), []
    for d in seq:
        k = str(Path(d).resolve())
        if k not in seen:
            seen.add(k)
            out.append(Path(d))
    return out


if not BUNDLE:
    zips, bundles = _walk_zips_and_bundles()
    bundles = _dedup(bundles)

    # 压缩包和已解压目录同时存在时**不许猜**：上一轮留下的旧目录会让这一轮
    # 静默训在旧数据上（看上去一切正常）。宁可停下让你删一个。
    both = bool(zips) and bool(bundles)
    if both:
        print("发现压缩包：", [z.name for z in zips])
        print("发现已解压目录：", [b.name for b in bundles])
        raise RuntimeError(
            "这个目录里**既有压缩包、又有解压好的包目录**，工具不敢替你猜用哪个。\n"
            "猜错会让你用上一轮的旧数据训练，而且不会报错。\n\n"
            "请二选一：\n"
            "  - 要用这次上传的压缩包：把解压出来的目录删掉，重跑本格。\n"
            "  - 要用已解压的目录：把 .zip 删掉，或直接在第 1 节把 BUNDLE 写成 "
            "目录名（%s 之一）。\n\n"
            "不确定的话，两个都删掉重新上传压缩包最稳。"
            % ", ".join(b.name for b in bundles))

    if zips and not bundles:
        print("发现的压缩包：", [str(z) for z in zips])
        if len(zips) == 1:
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(str(zips[0].parent))
            print("已解压：", zips[0])
            _, bundles = _walk_zips_and_bundles()
            bundles = _dedup(bundles)
        else:
            print("有多个压缩包，请在第 1 节 BUNDLE 里直接写解压出来的目录名，不要自动解压。")

    print("含 MANIFEST.json 的候选目录：", [str(b) for b in bundles])
    if len(bundles) == 1:
        BUNDLE = str(bundles[0])
    elif len(bundles) > 1:
        print("有多个候选，请在第 1 节 BUNDLE 里写清楚用哪一个。")

assert BUNDLE, ("请在上一格把 BUNDLE 写成解压出来的包目录名"
                "（就是含 MANIFEST.json 的那个目录）。")
BUNDLE = str(Path(BUNDLE).resolve())
assert Path(BUNDLE).is_dir(), "BUNDLE 不是目录：%s" % BUNDLE
assert (Path(BUNDLE) / "MANIFEST.json").is_file(), \
    "这个目录里没有 MANIFEST.json，多半不是你该训练的那个包：%s" % BUNDLE

WORK_DIR = str((Path.cwd() / WORK_DIR).resolve())
CFG = C.load_config(None)
CFG["workdir"] = WORK_DIR

# 母本守卫：工作目录落在母本根之下时，**在 mkdir 之前**就拒绝，绝不先建目录再判。
try:
    C.guard_master_write(WORK_DIR, CFG.get("master_roots"), False, "创建工作目录")
except C.MasterGuardError as exc:
    raise RuntimeError(
        "%s\n\n工作目录不能落在母本根之下。请把第 1 节的 WORK_DIR 改成"
        "一个不在 %s 之下的目录。" % (exc, CFG.get("master_roots")))

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

print("训练包   :", BUNDLE)
print("工作目录 :", WORK_DIR)
print("联系人在 pt_config.json 的 contact 里：", CFG.get("contact"))


## 4. 体检（必须做，不通过就停）

环境 / 输入 / 标签 / 契约四组。任一组不通过 = 停。有「判不了」的项 = 也要你确认过才继续。


In [ ]:
# ===== 4. 体检 =====
rep = pt_doctor.run_doctor(BUNDLE, CFG, None, str(Path(WORK_DIR) / "doctor.json"))
verdict = rep.overall()
DOCTOR_VERDICT = verdict
GATE_OK = False                    # 默认不放行；满足条件才置 True
print("\n体检总判：", verdict)

if verdict == "FAIL":
    raise RuntimeError(
        "体检不通过，请不要继续往下跑。\n"
        "把上面的内容（或 %s）发回本机。" % (Path(WORK_DIR) / "doctor.json"))

if verdict == "UNKNOWN" and not I_READ_THE_UNKNOWNS:
    raise RuntimeError(
        "体检有「判不了」的项。这些是**没检查到**的，不是没问题的。\n"
        "请把上面「判不了」那几条读完；确认无误后，回到第 1 节把 "
        "I_READ_THE_UNKNOWNS 改成 True，再重跑这一格。")

GATE_OK = True
print("体检放行，可以继续。")


## 5. 冒烟（CPU，小窗口，只写临时目录）

验证代码链路和你的环境是好的。不产模型，不写训练包。这一步不需要 GPU。


In [ ]:
# ===== 5. 冒烟 =====
assert globals().get("GATE_OK") is True, (
    "体检没有通过（或这一轮根本没跑体检），本格拒绝执行。\n"
    "请回到第 4 节把体检跑到「放行」。")

smoke = pt_train.run_one(BUNDLE, WORK_DIR, "smoke_" + time.strftime("%Y%m%d_%H%M%S"),
                         "smoke", None, CFG)

# 先说清「驱动不了」和「冒烟失败」的区别：前者是这个包的入口不认本工具要用的开关，
# 重跑多少次都是同一句话；后者才是这一轮跑坏了。混成一句，会把操作者支去发日志、白等一轮。
if smoke.get("capability_fail"):
    print(C.human_block(
        "这个包驱动不了（不是冒烟失败）",
        smoke.get("error") or "",
        "本工具按入口自己声明的开关来跑，不替它猜。缺的开关补不上，这一轮就起不来。",
        "把这一整段发回本机（连同包名）。不要重跑，也不要改包。",
        CFG.get("contact", C.DEFAULT_CONFIG["contact"])))
    raise RuntimeError("这个包驱动不了，已停止（不是冒烟失败）。")

bad = ("error" in smoke) or ("blocked" in smoke) or (smoke.get("rc") != 0)
if bad:
    print("---- 输出尾部 ----")
    print((smoke.get("stdout_tail") or "")[-1500:])
    print((smoke.get("stderr_tail") or "")[-1500:])
    print("---- 日志 ----", smoke.get("log"))
    raise RuntimeError("冒烟失败，已停止。把上面的输出发回本机，不要自己重跑或改包。")
print("冒烟通过。日志：", smoke.get("log"))


## 6. 真训练（需要 GPU）

默认关闭。要做真训练，回到第 1 节把 `DO_REAL_TRAIN` 改成 `True`，再重跑这一格。
没有 GPU 时会直接拒绝 —— 模型必须在 GPU 上训，这是硬规矩。


In [ ]:
# ===== 6. 真训练 =====
assert globals().get("GATE_OK") is True, (
    "体检没有通过（或这一轮根本没跑体检），本格拒绝执行。真训练会占 GPU、产模型，"
    "体检没过时绝不能跑。请回到第 4 节。")

if not DO_REAL_TRAIN:
    print("DO_REAL_TRAIN = False，已跳过真训练。")
    print("要真训练：回到第 1 节把它改成 True，再重跑这一格。")
else:
    gpu = C.gpu_info()
    print("GPU:", gpu)
    if not gpu.get("cuda_available"):
        raise RuntimeError(
            "没有可用的 CUDA 设备，拒绝真训练。\n"
            "模型必须在 GPU 上训练是硬规矩，CPU 上跑出来的模型不算数。\n"
            "可以先跑第 5 节冒烟验证链路；要真训练请换一台有 GPU 的机器。")

    seeds = CFG.get("seeds") or list(C.DEFAULT_CONFIG["seeds"])
    # 入口不认 --seed 时，多跑几个种子得到的是**同一条随机流**，它们的「离散度」不是噪声底。
    # 与其跑完再解释，不如当场只跑一个 —— 省 GPU，也不给人一个看起来像噪声底的数。
    _cap = pt_train.probe_entry_flags(Path(BUNDLE) / "train_from_xy.py")
    if len(seeds) > 1 and "--seed" not in (_cap.get("flags") or {}):
        print("(注记) 这个包的入口不认 --seed（它认的是：%s）。"
              % " ".join(sorted(_cap.get("flags") or {})))
        print("        多跑几个种子拿不到不同的随机流，已改为只跑 1 个。")
        print("        这样就没有噪声底 —— 第 8 节会照实报「判不了」，不会给一个假的底。")
        seeds = [seeds[0]]
    print("随机种子:", seeds, "（多跑几个种子是为了估计噪声底）")
    runs = []
    for s in seeds:
        tag = "%s_week_s%d_%s" % (Path(BUNDLE).name, s, time.strftime("%Y%m%d_%H%M%S"))
        print("\n>>> 种子", s, "试次", tag)
        r = pt_train.run_one(BUNDLE, WORK_DIR, tag, "week", s, CFG)
        runs.append(r)
        print("    rc=%s  decision=%s" % (r.get("rc"), r.get("decision")))
        if r.get("capability_fail"):
            print(C.human_block(
                "这个包驱动不了（不是训练失败）",
                r.get("error") or "",
                "本工具按入口自己声明的开关来跑，不替它猜。缺的开关补不上，这一轮就起不来。",
                "把这一整段发回本机（连同包名）。不要重跑，也不要改包。",
                CFG.get("contact", C.DEFAULT_CONFIG["contact"])))
            raise RuntimeError("这个包驱动不了，已停止（不是训练失败）。")
        if r.get("error") or r.get("blocked") or r.get("rc") != 0:
            print("---- 输出尾部 ----")
            print((r.get("stdout_tail") or "")[-1500:])
            print((r.get("stderr_tail") or "")[-1500:])
            raise RuntimeError("训练失败，已停止。把上面的输出发回本机。")

    summary = {"bundle": BUNDLE, "seeds": seeds, "runs": runs,
               "noise_floor": pt_train._noise_floor(runs), "generated_at": C.now_iso()}
    fp = Path(WORK_DIR) / "fingerprint.json"
    fp.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\n指纹已写入:", fp)
    print("噪声底:", json.dumps(summary["noise_floor"], ensure_ascii=False))
    if not (summary["noise_floor"] or {}).get("columns"):
        print("(注记) 上面没有给出噪声底数字，原因写在 note 里。"
              "没有噪声底时，第 8 节会把「增量是否显著」判成判不了 —— 那是**照实说**，不是出错。")


## 7. 超参搜索（只有本机随包发了网格时才做）

这一节**大多数时候是跳过的**。只有本机打这个包的时候附带了一份搜索网格，它才会真的跑。
屏幕上出现「本包没有随包网格，跳过这一节」，那是**正常跳过**，不是出错，继续往下跑就行。

要扫哪些参数、各扫哪些取值，都是本机定的（写在网格里）。你**不要**改它，也不要自己写一份 ——
写错不会有报错，只会得到一张每格数字都一样的表，那比没有更坏。

这一节总是**先打印一个计划**（要跑几格、几个种子、大概多久），然后才可能真跑。
真跑很贵：训练次数 = 格子数 × 种子数 + 1。所以第 1 节里有个开关 `DO_SEARCH`，默认关着。


In [ ]:
# ===== 7. 超参搜索 =====
assert globals().get("GATE_OK") is True, (
    "体检没有通过，本格拒绝执行。搜索要在 GPU 上跑很多次训练，体检没过时绝不能跑。")

_grid = Path(BUNDLE) / "SEARCH_GRID.json"
if not _grid.is_file():
    print("本包没有随包网格，跳过这一节（这是正常的，不是出错）。")
    print("搜索网格由本机随包发来；若本机明确说过给了，请把它放回包目录下再重跑这一格。")
else:
    print("网格：", _grid)
    print("\n---- 计划（只看不跑）----")
    _plan_rc = pt_search.cmd_search(["--bundle", BUNDLE, "--out", WORK_DIR])
    print("---- 计划结束 ----\n")
    if _plan_rc != 0:
        raise RuntimeError("搜索计划这一步就没过，把上面的内容发回本机。")

    if not DO_SEARCH:
        print("DO_SEARCH = False，已跳过真搜索。")
        print("看过上面的次数与预估时长、确认可以接受，再回第 1 节把它改成 True 重跑这一格。")
    else:
        print("开始真搜索（1 个并行）。跑完得到 surface.csv / region.json / trials.csv。")
        _rc = pt_search.cmd_search(["--bundle", BUNDLE, "--out", WORK_DIR,
                                    "--jobs", "1", "--commit"])
        if _rc != 0:
            raise RuntimeError("搜索没有正常跑完，把上面的内容发回本机。")
        print("\n搜索跑完了。下面这三份产物里**没有**『最高分参数』这种东西：")
        print("本工具报的是**邻域稳定区**（一段连成片的、彼此差不多好的区域），")
        print("并且会另跑一次 holdout 复核；复核没过时它会直接说没过。")
        print("你不需要解读，也不用挑参数，原样发回本机就行。")


## 8. 评估与裁决

把训练结果算成结论：复算门禁、量化噪声底、逐项过上线前置清单，最后给出一个**裁决姿态**。

这一节**不做任何改动**：不动包、不动线上模型、不往生产目录写东西。它只产两份报告。

它**不受体检闸门限制**，会照常跑完 —— 因为体检没过时，你更需要把「为什么没过」评估出来发回去。


In [ ]:
# ===== 8. 评估与裁决 =====
_no_runs = not (Path(WORK_DIR) / "fingerprint.json").is_file()
if _no_runs:
    print("注意：这个工作目录里没有训练记录，说明真训练那一节没跑成（或没打开开关）。")
    print("所以本次**没有可评估的训练结果** —— 下面的「不通过」说的是「没东西可评」，")
    print("不是说这个包不好。要出结论，请先回到第 6 节把真训练跑成。\n")

ev = pt_eval.run_eval(BUNDLE, WORK_DIR,
                      str(Path(WORK_DIR) / "eval.json"),
                      str(Path(WORK_DIR) / "report.md"))

# 用与第 4 节体检**同一个**渲染器出这一节：操作者上一格刚读完 [通过]/[判不了]，
# 这一格突然换成 [PASS]/[UNKNOWN] 是让人重新学一套记号；
# 更要紧的是"裁决姿态"那一项 —— 它恒为 PASS，印成「[PASS] 裁决姿态 blocked」
# 会让人把 blocked 读成通过。渲染器里它改印 [结论]。
print(ev.render())
print()
print("评估总判：", ev.overall())
print("裁决姿态：", ev.meta.get("posture"), "（本机要读的就是这一行，你原样发回即可）")

# 汇总按**全部**组来数：只列上面那三组的话，下面这两行会冒出
# 你在正文里根本没看到的项名（试次记录、训练产物完整……）。
_bad = ev.fails()
_unk = ev.unknowns()
print()
if _bad:
    print("有不通过的项 %d 个：%s" % (len(_bad), "、".join(c["name"] for c in _bad)))
if _unk:
    print("有判不了的项 %d 个（**不等于通过**）：%s"
          % (len(_unk), "、".join(c["name"] for c in _unk)))
print()
print("人读报告：", Path(WORK_DIR) / "report.md")
print("机器可读：", Path(WORK_DIR) / "eval.json")
print("请把这两份**原样发回本机**，不要自己判断结果好坏，也不要改任何数字。")


## 9. 打包回传

生成一个 zip，在左边文件列表里右键 `Download` 发给本机。**不要自己判断结果好坏**，
不要往生产目录覆盖任何东西，裁决由本机做。


In [ ]:
# ===== 9. 打包回传 =====
# 这一格**无论如何都跑**：体检没通过时也要把 doctor.json 发回本机，
# 那正是本机需要看到的东西。包名里带着这一轮的结论，本机一眼就能分辨。
#
# 这一格调用**工具自己的 collect**，不另写一份打包逻辑。
# 另写一份的后果已经实测到了：zip 旁边那个 .sha256 会没有
# （而本机要求你连它一起发回），逐件写明"哪些大件没带、为什么"的
# COLLECT.json 也会没有 —— 那两样正是"不会悄悄丢东西"的凭据。
_argv = ["--workdir", WORK_DIR, "--out", ".", "--bundle", BUNDLE]
if WITH_MODELS:
    _argv.append("--with-models")
_rc = pt_pack.cmd_collect(_argv)
print()
if _rc != 0:
    print("注意：这一节没能正常生成回传包（返回码 %s）。" % _rc)
    print("请把这一格的全部输出**原样复制**发回本机；不要自己重跑，也不要改包。")
else:
    print("请把 zip 与同名的 .sha256 **一起** Download，发回本机。")
    print("zip 里的 COLLECT.json 逐件列明了哪些大件没带、为什么没带 —— 那不是丢件。")


## 出问题时

- 任何 `!!!!` 方框都是说给你看的，照着「下一步做什么」做，并把原文发回本机。
- 日志在 `_pt_work/logs/` 下，每个试次一个 `.log`。
- 不确定的事**不要猜**，也不要把「判不了」当成「通过」。
